# 🚗 Estimation du Prix des Véhicules d'Occasion — Car Dekho
**Étudiant :** Meimoune Baba Cheikh Sidiya  
**Matricule :** C34620  
**Professeur :** Ezyn SEGNANE  
**Type de tâche :** Régression supervisée  
**Dataset :** Vehicle Dataset (CarDekho) — multi-villes fusionné


## 1. Imports et Configuration

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("teal")
print("Imports OK ✅")


## 2. Chargement et Exploration des Données (EDA)

In [ ]:

# Chargement du dataset nettoyé (fusion multi-villes déjà effectuée)
df_raw = pd.read_csv('car_dekho_cleaned.csv')

# Correction des colonnes booléennes
bool_cols = ['fuel_Diesel','fuel_Electric','fuel_LPG','fuel_Petrol',
             'seller_type_Individual','seller_type_Trustmark Dealer','transmission_Manual']
for c in bool_cols:
    df_raw[c] = df_raw[c].astype(int)

print("Shape:", df_raw.shape)
print("\nColonnes:", df_raw.columns.tolist())
df_raw.head()


In [ ]:

# Statistiques descriptives
df_raw.describe()


In [ ]:

# Valeurs manquantes
print("Valeurs manquantes:")
print(df_raw.isnull().sum())
print("\nTypes des colonnes:")
print(df_raw.dtypes)


## 3. Analyse Exploratoire (EDA) — Visualisations

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribution des variables numériques', fontsize=16, fontweight='bold')

# Prix de vente (variable cible)
axes[0,0].hist(df_raw['selling_price'], bins=50, color='#0D9488', edgecolor='white')
axes[0,0].set_title('Distribution du Prix de Vente')
axes[0,0].set_xlabel('Prix (₹)')

# Kilomètres parcourus
axes[0,1].hist(df_raw['km_driven'], bins=50, color='#3B82F6', edgecolor='white')
axes[0,1].set_title('Distribution Kilométrage')
axes[0,1].set_xlabel('km_driven (standardisé)')

# Année
axes[0,2].hist(df_raw['year'], bins=30, color='#8B5CF6', edgecolor='white')
axes[0,2].set_title("Distribution Année (standardisée)")
axes[0,2].set_xlabel('year (standardisé)')

# Âge du véhicule
axes[1,0].hist(df_raw['car_age'], bins=30, color='#F59E0B', edgecolor='white')
axes[1,0].set_title("Âge du Véhicule")
axes[1,0].set_xlabel('car_age (standardisé)')

# Prix par transmission
trans = ['Manuel', 'Automatique']
prices_trans = [
    df_raw[df_raw['transmission_Manual']==1]['selling_price'],
    df_raw[df_raw['transmission_Manual']==0]['selling_price']
]
axes[1,1].boxplot(prices_trans, labels=trans, patch_artist=True)
axes[1,1].set_title('Prix par Type de Transmission')
axes[1,1].set_ylabel('Prix (₹)')

# Prix par type de carburant
fuel_types = ['Petrol', 'Diesel', 'Electric', 'LPG', 'CNG']
fuel_prices = []
for f in fuel_types:
    col = f'fuel_{f}'
    if col in df_raw.columns:
        fuel_prices.append(df_raw[df_raw[col]==1]['selling_price'])
    else:
        fuel_prices.append(df_raw[(df_raw['fuel_Petrol']==0) & (df_raw['fuel_Diesel']==0) &
                                   (df_raw['fuel_Electric']==0) & (df_raw['fuel_LPG']==0)]['selling_price'])
axes[1,2].boxplot(fuel_prices, labels=fuel_types, patch_artist=True)
axes[1,2].set_title('Prix par Type de Carburant')
axes[1,2].set_ylabel('Prix (₹)')

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée ✅")


In [ ]:

# Matrice de corrélation
plt.figure(figsize=(12, 8))
corr = df_raw.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', 
            cmap='RdYlGn', center=0, linewidths=0.5,
            cbar_kws={'label': 'Corrélation'})
plt.title('Matrice de Corrélation — Variables du Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Corrélation la plus forte avec selling_price:")
print(corr['selling_price'].sort_values(ascending=False))


In [ ]:

# Scatter plots : prix vs variables clés
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Relations entre variables et prix de vente', fontsize=14, fontweight='bold')

axes[0].scatter(df_raw['year'], df_raw['selling_price'], alpha=0.3, color='#0D9488', s=10)
axes[0].set_xlabel('Année (standardisée)'); axes[0].set_ylabel('Prix (₹)')
axes[0].set_title('Année vs Prix')

axes[1].scatter(df_raw['km_driven'], df_raw['selling_price'], alpha=0.3, color='#3B82F6', s=10)
axes[1].set_xlabel('Kilométrage (standardisé)'); axes[1].set_ylabel('Prix (₹)')
axes[1].set_title('Kilométrage vs Prix')

axes[2].scatter(df_raw['brand_encoded'], df_raw['selling_price'], alpha=0.3, color='#F59E0B', s=10)
axes[2].set_xlabel('Marque (encodée)'); axes[2].set_ylabel('Prix (₹)')
axes[2].set_title('Marque vs Prix')

plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Préparation des Données — Train/Test Split

In [ ]:

# Séparation features / target
feature_cols = [c for c in df_raw.columns if c != 'selling_price']
X = df_raw[feature_cols]
y = df_raw['selling_price']

print(f"Features: {X.shape} | Target: {y.shape}")
print(f"Features utilisées: {feature_cols}")

# Séparation stratifiée (bins du prix)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTrain: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# KFold pour validation croisée
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print("\nKFold CV (k=5) configuré ✅")


## 5. Modélisation — 4 Algorithmes Comparés

### 5.1 Régression Linéaire (Baseline)

In [ ]:

# ─ Régression Linéaire ─
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

r2_lr   = r2_score(y_test, y_pred_lr)
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
cv_lr   = cross_val_score(lr, X, y, cv=kf, scoring='r2')

print("=" * 50)
print("RÉGRESSION LINÉAIRE")
print("=" * 50)
print(f"R²   (test)  : {r2_lr:.4f}")
print(f"MAE  (test)  : {mae_lr:,.0f} ₹")
print(f"RMSE (test)  : {rmse_lr:,.0f} ₹")
print(f"CV R² (5-fold): {cv_lr.mean():.4f} ± {cv_lr.std():.4f}")
print("\nJustification: Modèle baseline simple. Suppose une relation")
print("linéaire entre les features et le prix. Utile comme référence.")


### 5.2 Random Forest + GridSearchCV

In [ ]:

# ─ Random Forest ─
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
}
rf_gs = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)
rf_gs.fit(X_train, y_train)
best_rf = rf_gs.best_estimator_
y_pred_rf = best_rf.predict(X_test)

r2_rf   = r2_score(y_test, y_pred_rf)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
cv_rf   = cross_val_score(best_rf, X, y, cv=kf, scoring='r2')

print("=" * 50)
print("RANDOM FOREST")
print("=" * 50)
print(f"Meilleurs hyperparamètres: {rf_gs.best_params_}")
print(f"R²   (test)  : {r2_rf:.4f}")
print(f"MAE  (test)  : {mae_rf:,.0f} ₹")
print(f"RMSE (test)  : {rmse_rf:,.0f} ₹")
print(f"CV R² (5-fold): {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")
print("\nJustification: Ensemble de arbres de décision, robuste aux outliers.")
print("GridSearchCV optimise n_estimators, max_depth, min_samples_split.")


### 5.3 XGBoost + GridSearchCV (Meilleur Modèle)

In [ ]:

# ─ XGBoost ─
xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
}
xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    xgb_param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)
xgb_gs.fit(X_train, y_train)
best_xgb = xgb_gs.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

r2_xgb   = r2_score(y_test, y_pred_xgb)
mae_xgb  = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
cv_xgb   = cross_val_score(best_xgb, X, y, cv=kf, scoring='r2')

print("=" * 50)
print("XGBOOST")
print("=" * 50)
print(f"Meilleurs hyperparamètres: {xgb_gs.best_params_}")
print(f"R²   (test)  : {r2_xgb:.4f}")
print(f"MAE  (test)  : {mae_xgb:,.0f} ₹")
print(f"RMSE (test)  : {rmse_xgb:,.0f} ₹")
print(f"CV R² (5-fold): {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}")
print("\nJustification: Gradient boosting régularisé. Optimise séquentiellement")
print("les arbres en corrigeant les erreurs du modèle précédent.")


### 5.4 SVR (Support Vector Regression) + GridSearchCV

In [ ]:

# ─ SVR ─
# SVR nécessite une normalisation supplémentaire
svr_scaler = StandardScaler()
X_train_svr = svr_scaler.fit_transform(X_train)
X_test_svr  = svr_scaler.transform(X_test)

svr_param_grid = {
    'C': [10, 100],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.1, 1.0],
}
svr_gs = GridSearchCV(
    SVR(kernel='rbf'),
    svr_param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)
svr_gs.fit(X_train_svr, y_train)
best_svr = svr_gs.best_estimator_
y_pred_svr = best_svr.predict(X_test_svr)

r2_svr   = r2_score(y_test, y_pred_svr)
mae_svr  = mean_absolute_error(y_test, y_pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test, y_pred_svr))
X_svr_full = svr_scaler.fit_transform(X)
cv_svr   = cross_val_score(SVR(kernel='rbf', C=svr_gs.best_params_['C'],
                               gamma=svr_gs.best_params_['gamma'],
                               epsilon=svr_gs.best_params_['epsilon']),
                           X_svr_full, y, cv=kf, scoring='r2')

print("=" * 50)
print("SVR (Support Vector Regression)")
print("=" * 50)
print(f"Meilleurs hyperparamètres: {svr_gs.best_params_}")
print(f"R²   (test)  : {r2_svr:.4f}")
print(f"MAE  (test)  : {mae_svr:,.0f} ₹")
print(f"RMSE (test)  : {rmse_svr:,.0f} ₹")
print(f"CV R² (5-fold): {cv_svr.mean():.4f} ± {cv_svr.std():.4f}")
print("\nNOTE: SVR sous-performe sur ce dataset hétérogène à grande échelle.")
print("Les prix varient de 20 000 à 1 200 000 ₹, ce qui défavorise SVR.")


## 6. Comparaison et Sélection du Meilleur Modèle

In [ ]:

# Tableau récapitulatif
results_df = pd.DataFrame({
    'Modèle': ['Régression Linéaire', 'Random Forest', 'XGBoost', 'SVR'],
    'R² Test': [r2_lr, r2_rf, r2_xgb, r2_svr],
    'MAE (₹)': [mae_lr, mae_rf, mae_xgb, mae_svr],
    'RMSE (₹)': [rmse_lr, rmse_rf, rmse_xgb, rmse_svr],
    'CV R² Moy.': [cv_lr.mean(), cv_rf.mean(), cv_xgb.mean(), cv_svr.mean()],
    'CV R² Std':  [cv_lr.std(),  cv_rf.std(),  cv_xgb.std(),  cv_svr.std()],
}).set_index('Modèle')

print("=" * 75)
print("TABLEAU COMPARATIF FINAL")
print("=" * 75)
print(results_df.to_string())

best_model_name = results_df['R² Test'].idxmax()
print(f"\n🏆 MEILLEUR MODÈLE : {best_model_name}")
print(f"   R² = {results_df.loc[best_model_name, 'R² Test']:.4f}")
print(f"   MAE = {results_df.loc[best_model_name, 'MAE (₹)']:,.0f} ₹")


In [ ]:

# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Comparaison des 4 Algorithmes — Car Dekho', fontsize=14, fontweight='bold')

models_list  = ['Régression\nLinéaire', 'Random\nForest', 'XGBoost', 'SVR']
colors = ['#94A3B8', '#3B82F6', '#0D9488', '#F59E0B']

# R²
r2_vals = [r2_lr, r2_rf, r2_xgb, max(r2_svr, 0)]
bars = axes[0].bar(models_list, [r2_lr, r2_rf, r2_xgb, r2_svr], color=colors, edgecolor='white')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title('R² (plus haut = meilleur)')
axes[0].set_ylabel('R²')
for bar, val in zip(bars, [r2_lr, r2_rf, r2_xgb, r2_svr]):
    axes[0].text(bar.get_x() + bar.get_width()/2., max(val, 0) + 0.01, 
                 f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# MAE
bars2 = axes[1].bar(models_list, [mae_lr, mae_rf, mae_xgb, mae_svr], color=colors, edgecolor='white')
axes[1].set_title('MAE — ₹ (plus bas = meilleur)')
axes[1].set_ylabel('MAE (₹)')
for bar, val in zip(bars2, [mae_lr, mae_rf, mae_xgb, mae_svr]):
    axes[1].text(bar.get_x() + bar.get_width()/2., val + 1000, 
                 f'{val:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=9)

# CV R²
cv_means = [cv_lr.mean(), cv_rf.mean(), cv_xgb.mean(), cv_svr.mean()]
cv_stds  = [cv_lr.std(),  cv_rf.std(),  cv_xgb.std(),  cv_svr.std()]
axes[2].bar(models_list, cv_means, color=colors, edgecolor='white', 
            yerr=cv_stds, capsize=5, ecolor='black')
axes[2].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('CV R² Moyen ± Std (k=5)')
axes[2].set_ylabel('CV R²')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegardé ✅")


## 7. Importance des Variables et Interprétation

In [ ]:

# Feature importance — XGBoost
fi = pd.Series(best_xgb.feature_importances_, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(fi.index, fi.values, color='#0D9488', edgecolor='white')
plt.xlabel('Importance')
plt.title('Importance des Variables — XGBoost (Meilleur Modèle)', fontsize=13, fontweight='bold')

for bar, val in zip(bars, fi.values):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2., 
             f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nInterprétation des variables les plus influentes:")
fi_top = fi.sort_values(ascending=False)
print(fi_top.to_string())


In [ ]:

# Graphique Réel vs Prédit — XGBoost
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter réel vs prédit
axes[0].scatter(y_test, y_pred_xgb, alpha=0.4, color='#0D9488', s=15)
min_val = min(y_test.min(), y_pred_xgb.min())
max_val = max(y_test.max(), y_pred_xgb.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prédiction parfaite')
axes[0].set_xlabel('Prix réel (₹)')
axes[0].set_ylabel('Prix prédit (₹)')
axes[0].set_title(f'XGBoost — Réel vs Prédit (R²={r2_xgb:.3f})')
axes[0].legend()

# Résidus
residuals = y_test - y_pred_xgb
axes[1].scatter(y_pred_xgb, residuals, alpha=0.4, color='#3B82F6', s=15)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Prix prédit (₹)')
axes[1].set_ylabel('Résidu (₹)')
axes[1].set_title('Analyse des Résidus — XGBoost')

plt.tight_layout()
plt.savefig('predictions_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Sauvegarde des Modèles

In [ ]:

# Sauvegarde de tous les modèles entraînés
joblib.dump(lr, 'model_lr.pkl')
joblib.dump(best_rf, 'model_rf.pkl')
joblib.dump(best_xgb, 'model_xgb.pkl')
joblib.dump(best_svr, 'model_svr.pkl')
joblib.dump(svr_scaler, 'scaler_svr.pkl')

print("Tous les modèles sauvegardés :")
print("  ✅ model_lr.pkl  — Régression Linéaire")
print("  ✅ model_rf.pkl  — Random Forest (tuné)")
print("  ✅ model_xgb.pkl — XGBoost (tuné) ← MEILLEUR")
print("  ✅ model_svr.pkl — SVR (tuné)")
print("  ✅ scaler_svr.pkl — Scaler pour SVR")


In [ ]:
# Sauvegarde des métriques et du meilleur modèle
import json

model_metrics = {
    "Régression Linéaire": {
        "r2": round(float(r2_lr), 4),
        "mae": round(float(mae_lr), 0),
        "rmse": round(float(rmse_lr), 0),
        "cv_r2_mean": round(float(cv_lr.mean()), 4),
        "cv_r2_std": round(float(cv_lr.std()), 4),
    },
    "Random Forest": {
        "r2": round(float(r2_rf), 4),
        "mae": round(float(mae_rf), 0),
        "rmse": round(float(rmse_rf), 0),
        "cv_r2_mean": round(float(cv_rf.mean()), 4),
        "cv_r2_std": round(float(cv_rf.std()), 4),
        "best_params": rf_gs.best_params_,
    },
    "XGBoost": {
        "r2": round(float(r2_xgb), 4),
        "mae": round(float(mae_xgb), 0),
        "rmse": round(float(rmse_xgb), 0),
        "cv_r2_mean": round(float(cv_xgb.mean()), 4),
        "cv_r2_std": round(float(cv_xgb.std()), 4),
        "best_params": xgb_gs.best_params_,
    },
    "SVR": {
        "r2": round(float(r2_svr), 4),
        "mae": round(float(mae_svr), 0),
        "rmse": round(float(rmse_svr), 0),
        "cv_r2_mean": round(float(cv_svr.mean()), 4),
        "cv_r2_std": round(float(cv_svr.std()), 4),
        "best_params": svr_gs.best_params_,
    },
}

with open("model_metrics.json", "w", encoding="utf-8") as f:
    json.dump(model_metrics, f, indent=2, ensure_ascii=False, default=float)

# Alias du meilleur modèle
joblib.dump(best_xgb, "model_best.pkl")

print("Métriques sauvegardées ✅  model_metrics.json")
print("model_best.pkl (alias XGBoost) sauvegardé ✅")


## 9. Conclusion et Perspectives

### 🏆 Résumé des performances

| Modèle | R² | MAE (₹) | CV R² |
|--------|-----|---------|-------|
| Régression Linéaire | ~0.59 | ~149 000 | 0.57 |
| Random Forest | ~0.73 | ~111 000 | 0.72 |
| **XGBoost** | **~0.75** | **~107 000** | **0.75** |
| SVR | ~-0.01 | ~224 000 | -0.01 |

### 📌 Variables les plus influentes
1. **transmission_Manual** (31.6%) — Les voitures manuelles vs automatiques ont un impact majeur sur le prix
2. **year** (26.1%) — L'année de fabrication détermine fortement la valeur résiduelle
3. **fuel_Diesel** (24.2%) — Les moteurs diesel sont valorisés différemment

### 🔍 Analyse critique
- **XGBoost** est le meilleur modèle : robuste, stable (CV std ≈ 0.01) et interprétable via feature importance
- **SVR** échoue sur ce dataset car il est sensible aux grandes variations d'échelle (prix de 20k à 1.2M ₹)
- **Random Forest** est une bonne alternative, moins sensible aux hyperparamètres

### 🚀 Perspectives d'amélioration
- Ajout de données supplémentaires (moteur, puissance, consommation)
- Feature engineering : ratio prix/km, catégorie premium/budget
- Modèles d'ensemble : stacking LR + RF + XGBoost
- Déploiement via Streamlit Cloud (cf. app.py)
